# Day 10: Netflix Movies and TV Shows Dataset Analysis
We implement a complete Exploratory Data Analysis (EDA) on the Netflix dataset following the 15-step standard workflow.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12


### Step 1: Load Dataset


In [ ]:
# Load dataset
df = pd.read_csv("../../DataSet/netflix_titles.csv")
print("Netflix dataset loaded successfully!")


### Step 2: Check Shape


In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.shape


### Step 3: View Dataset Information


In [ ]:
# Check first few records
df.head()


In [ ]:
# Check random sample
df.sample(5)


### Step 4: Check Data Types


In [ ]:
df.info()


In [ ]:
df.dtypes


### Step 5: Identify Missing Values


In [ ]:
# Check missing count and percentage
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100

null_df = pd.DataFrame({'Null Count': null_counts, 'Percentage (%)': null_pct})
print("Missing Value Summary:")
null_df[null_df['Null Count'] > 0]


### Step 6: Handle Missing Values
* **director**: ~30% missing values. We will fill with `"Unknown"`.
* **cast**: ~9% missing. We will fill with `"Unknown"`.
* **country**: ~6% missing. We will fill with the **mode** (most frequent country).
* **date_added**: ~0.1% missing. We will fill with the mode.
* **rating**: ~0.08% missing. We will fill with `"UR"` (Unrated).


In [ ]:
# 1. Fill missing director and cast with 'Unknown'
df["director"] = df["director"].fillna("Unknown")
df["cast"] = df["cast"].fillna("Unknown")

# 2. Fill country and date_added with mode
df["country"] = df["country"].fillna(df["country"].mode()[0])
df["date_added"] = df["date_added"].fillna(df["date_added"].mode()[0])

# 3. Fill rating with 'UR'
df["rating"] = df["rating"].fillna("UR")

# Verify all missing values are handled
print("Remaining null counts:")
df.isnull().sum()


### Step 7 & 8: Check and Remove Duplicates


In [ ]:
print("Duplicate count:", df.duplicated().sum())
df.drop_duplicates(inplace=True)


### Step 9: Generate Statistical Summary


In [ ]:
# Statistical summary of numeric fields (e.g., release_year)
df.describe()


In [ ]:
# Summary of categorical columns
df.describe(include="object")


### Step 10: Outlier Detection


In [ ]:
# Detect outlier years in the release_year column using a boxplot
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["release_year"], color="lightcoral")
plt.title("Boxplot of Release Year")
plt.xlabel("Release Year")
plt.show()

# Identify how many movies/shows are released before 1990
older_shows = df[df["release_year"] < 1990]
print(f"Number of movies/shows released before 1990: {len(older_shows)}")


### Step 11: Perform Univariate Analysis


In [ ]:
# 1. Type Distribution (Movie vs TV Show)
type_counts = df["type"].value_counts()
print(type_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Bar plot
sns.countplot(x="type", data=df, ax=axes[0], palette="Set2")
axes[0].set_title("Content Type Counts")
axes[0].set_xlabel("Type")
axes[0].set_ylabel("Count")

# Pie chart
axes[1].pie(type_counts, labels=type_counts.index, autopct="%1.1f%%", startangle=90, colors=["#66b3ff", "#ffcc99"])
axes[1].set_title("Content Type Ratio")
plt.suptitle("Netflix Content Distribution", fontsize=16)
plt.show()


In [ ]:
# 2. Top 10 Countries producing content
top_countries = df["country"].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_countries.values, y=top_countries.index, palette="viridis")
plt.title("Top 10 Content Producing Countries")
plt.xlabel("Count")
plt.ylabel("Country")
plt.show()


In [ ]:
# 3. Rating Distribution
rating_counts = df["rating"].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=rating_counts.index, y=rating_counts.values, palette="rocket")
plt.title("Top 10 Ratings Distribution")
plt.xlabel("Rating Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


### Step 12: Perform Bivariate Analysis


In [ ]:
# Content Type vs Release Year
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x="release_year", hue="type", fill=True, common_norm=False, palette="Set1", alpha=0.5)
plt.title("Distribution of Release Years by Type")
plt.xlabel("Release Year")
plt.ylabel("Density")
plt.xlim(1980, 2021) # Zoom in to recent decades
plt.show()


### Step 13: Correlation Analysis


In [ ]:
# Let's map Movie/TV Show to numerical (0 and 1) to inspect simple correlations
df_corr = df.copy()
df_corr["type_num"] = df_corr["type"].map({"Movie": 0, "TV Show": 1})

corr = df_corr[["release_year", "type_num"]].corr()
print("Correlation Matrix:")
display(corr)

sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix Heatmap")
plt.show()


### Step 14: Feature Engineering


In [ ]:
# 1. Parse date_added to extract Year and Month added
df["date_added"] = df["date_added"].str.strip()
df["year_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y", errors="coerce").dt.year
df["month_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y", errors="coerce").dt.month

# 2. Extract Primary Genre (first genre listed in 'listed_in' column)
df["primary_genre"] = df["listed_in"].apply(lambda x: x.split(",")[0].strip())

print(df[["title", "year_added", "month_added", "primary_genre"]].head())

# Visualizing Primary Genre counts for Movies vs TV Shows
top_genres = df["primary_genre"].value_counts().head(10).index
plt.figure(figsize=(10, 6))
sns.countplot(y="primary_genre", hue="type", data=df[df["primary_genre"].isin(top_genres)], palette="muted")
plt.title("Top 10 Genres on Netflix (by Content Type)")
plt.xlabel("Count")
plt.ylabel("Genre")
plt.show()


### Step 15: Draw Conclusions and Insights
1. **Dominance of Movies**: Movies make up approximately 69.1% of Netflix's content library, while TV Shows comprise the remaining 30.9%.
2. **Global Leader**: The United States is by far the leading country for producing Netflix content, followed by India and the United Kingdom.
3. **Target Audience**: The majority of content is rated TV-MA (Mature Audiences) and TV-14 (Parents Strongly Cautioned), highlighting a strong focus on older teen and adult viewers.
4. **Recent Surge**: The density plot of release years reveals a exponential surge in content releases beginning around 2010, peaking around 2018-2020.
